In [1]:
!pip install pyspark -q

In [5]:
# Import Libraries and Start Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, round, desc, avg, sum as spark_sum

spark = SparkSession.builder.appName("CourseAnalysis").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [6]:
# Load Large Enrollment Data
enrollment_data = [
    (1, 101, "Data Science"), (2, 101, "Data Science"), (3, 102, "Machine Learning"),
    (4, 103, "Web Development"), (5, 102, "Machine Learning"), (6, 104, "Cloud Computing"),
    (7, 101, "Data Science"), (8, 103, "Web Development"), (9, 104, "Cloud Computing"),
    (10, 102, "Machine Learning"), (11, 103, "Web Development"), (12, 104, "Cloud Computing"),
    (13, 101, "Data Science"), (14, 102, "Machine Learning"), (15, 103, "Web Development"),
    (16, 105, "Cybersecurity"), (17, 105, "Cybersecurity"), (18, 106, "Deep Learning"),
    (19, 106, "Deep Learning"), (20, 105, "Cybersecurity"), (21, 106, "Deep Learning"),
    (22, 107, "DevOps"), (23, 107, "DevOps"), (24, 107, "DevOps"), (25, 108, "Blockchain"),
    (26, 108, "Blockchain"), (27, 109, "NLP"), (28, 109, "NLP"), (29, 110, "AR/VR"),
    (30, 110, "AR/VR")
]

enrollment_df = spark.createDataFrame(enrollment_data, ["student_id", "course_id", "course_name"])
print("Enrollment Data:")
enrollment_df.show()
print("Total Enrollments:", enrollment_df.count())

Enrollment Data:
+----------+---------+----------------+
|student_id|course_id|     course_name|
+----------+---------+----------------+
|         1|      101|    Data Science|
|         2|      101|    Data Science|
|         3|      102|Machine Learning|
|         4|      103| Web Development|
|         5|      102|Machine Learning|
|         6|      104| Cloud Computing|
|         7|      101|    Data Science|
|         8|      103| Web Development|
|         9|      104| Cloud Computing|
|        10|      102|Machine Learning|
|        11|      103| Web Development|
|        12|      104| Cloud Computing|
|        13|      101|    Data Science|
|        14|      102|Machine Learning|
|        15|      103| Web Development|
|        16|      105|   Cybersecurity|
|        17|      105|   Cybersecurity|
|        18|      106|   Deep Learning|
|        19|      106|   Deep Learning|
|        20|      105|   Cybersecurity|
+----------+---------+----------------+
only showing top 20 row

In [7]:
# Load Large Progress Data
progress_data = [
    (1, 101, "completed"), (2, 101, "dropped"), (3, 102, "completed"),
    (4, 103, "completed"), (5, 102, "dropped"), (6, 104, "completed"),
    (7, 101, "completed"), (8, 103, "dropped"), (9, 104, "completed"),
    (10, 102, "completed"), (11, 103, "dropped"), (12, 104, "completed"),
    (13, 101, "dropped"), (14, 102, "completed"), (15, 103, "completed"),
    (16, 105, "completed"), (17, 105, "dropped"), (18, 106, "completed"),
    (19, 106, "dropped"), (20, 105, "completed"), (21, 106, "completed"),
    (22, 107, "dropped"), (23, 107, "dropped"), (24, 107, "completed"),
    (25, 108, "dropped"), (26, 108, "dropped"), (27, 109, "completed"),
    (28, 109, "completed"), (29, 110, "dropped"), (30, 110, "completed")
]

progress_df = spark.createDataFrame(progress_data, ["student_id", "course_id", "status"])
print("Progress Data:")
progress_df.show()
print("Total Progress Records:", progress_df.count())

Progress Data:
+----------+---------+---------+
|student_id|course_id|   status|
+----------+---------+---------+
|         1|      101|completed|
|         2|      101|  dropped|
|         3|      102|completed|
|         4|      103|completed|
|         5|      102|  dropped|
|         6|      104|completed|
|         7|      101|completed|
|         8|      103|  dropped|
|         9|      104|completed|
|        10|      102|completed|
|        11|      103|  dropped|
|        12|      104|completed|
|        13|      101|  dropped|
|        14|      102|completed|
|        15|      103|completed|
|        16|      105|completed|
|        17|      105|  dropped|
|        18|      106|completed|
|        19|      106|  dropped|
|        20|      105|completed|
+----------+---------+---------+
only showing top 20 rows
Total Progress Records: 30


In [8]:
# Join Enrollment and Progress Tables to Get Course-Wise Progress
joined_df = enrollment_df.join(progress_df, on=["student_id", "course_id"], how="left")
print("Joined Data:")
joined_df.show()
print("Total Records after Join:", joined_df.count())

Joined Data:
+----------+---------+----------------+---------+
|student_id|course_id|     course_name|   status|
+----------+---------+----------------+---------+
|         5|      102|Machine Learning|  dropped|
|         4|      103| Web Development|completed|
|         2|      101|    Data Science|  dropped|
|         7|      101|    Data Science|completed|
|         9|      104| Cloud Computing|completed|
|         8|      103| Web Development|  dropped|
|         3|      102|Machine Learning|completed|
|        12|      104| Cloud Computing|completed|
|        13|      101|    Data Science|  dropped|
|         6|      104| Cloud Computing|completed|
|        11|      103| Web Development|  dropped|
|         1|      101|    Data Science|completed|
|        15|      103| Web Development|completed|
|        10|      102|Machine Learning|completed|
|        14|      102|Machine Learning|completed|
|        19|      106|   Deep Learning|  dropped|
|        25|      108|      Blockchai

In [9]:
# Group By Course and Count Total Enrolled and Completed Students
course_stats = joined_df.groupBy("course_id", "course_name").agg(
    count("student_id").alias("total_enrolled"),
    count(when(col("status") == "completed", True)).alias("total_completed"),
    count(when(col("status") == "dropped", True)).alias("total_dropped")
)
print("Course-wise Stats:")
course_stats.show()

Course-wise Stats:
+---------+----------------+--------------+---------------+-------------+
|course_id|     course_name|total_enrolled|total_completed|total_dropped|
+---------+----------------+--------------+---------------+-------------+
|      107|          DevOps|             3|              1|            2|
|      102|Machine Learning|             4|              3|            1|
|      101|    Data Science|             4|              2|            2|
|      103| Web Development|             4|              2|            2|
|      105|   Cybersecurity|             3|              2|            1|
|      110|           AR/VR|             2|              1|            1|
|      109|             NLP|             2|              2|            0|
|      104| Cloud Computing|             3|              3|            0|
|      108|      Blockchain|             2|              0|            2|
|      106|   Deep Learning|             3|              2|            1|
+---------+--------

In [10]:
# Calculate Completion Rate and Dropout Rate Per Course
course_stats_with_rate = course_stats.withColumn(
    "completion_rate_%", round((col("total_completed") / col("total_enrolled")) * 100, 2)
).withColumn(
    "dropout_rate_%", round((col("total_dropped") / col("total_enrolled")) * 100, 2)
)
print("Course Stats with Rates:")
course_stats_with_rate.show()

Course Stats with Rates:
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|course_id|     course_name|total_enrolled|total_completed|total_dropped|completion_rate_%|dropout_rate_%|
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|      107|          DevOps|             3|              1|            2|            33.33|         66.67|
|      102|Machine Learning|             4|              3|            1|             75.0|          25.0|
|      101|    Data Science|             4|              2|            2|             50.0|          50.0|
|      103| Web Development|             4|              2|            2|             50.0|          50.0|
|      105|   Cybersecurity|             3|              2|            1|            66.67|         33.33|
|      110|           AR/VR|             2|              1|            1|             50.0|          50.0|
|      109| 

In [11]:
# Output Top Completed Courses
print("Top Completed Courses:")
course_stats_with_rate.orderBy(desc("total_completed")).show()

Top Completed Courses:
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|course_id|     course_name|total_enrolled|total_completed|total_dropped|completion_rate_%|dropout_rate_%|
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|      102|Machine Learning|             4|              3|            1|             75.0|          25.0|
|      104| Cloud Computing|             3|              3|            0|            100.0|           0.0|
|      101|    Data Science|             4|              2|            2|             50.0|          50.0|
|      103| Web Development|             4|              2|            2|             50.0|          50.0|
|      105|   Cybersecurity|             3|              2|            1|            66.67|         33.33|
|      109|             NLP|             2|              2|            0|            100.0|           0.0|
|      106|   

In [12]:
# Output Top Dropped-Out Courses
print("Top Dropped-Out Courses:")
course_stats_with_rate.orderBy(desc("total_dropped")).show()

Top Dropped-Out Courses:
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|course_id|     course_name|total_enrolled|total_completed|total_dropped|completion_rate_%|dropout_rate_%|
+---------+----------------+--------------+---------------+-------------+-----------------+--------------+
|      107|          DevOps|             3|              1|            2|            33.33|         66.67|
|      101|    Data Science|             4|              2|            2|             50.0|          50.0|
|      103| Web Development|             4|              2|            2|             50.0|          50.0|
|      108|      Blockchain|             2|              0|            2|              0.0|         100.0|
|      102|Machine Learning|             4|              3|            1|             75.0|          25.0|
|      105|   Cybersecurity|             3|              2|            1|            66.67|         33.33|
|      110| 

In [13]:
# Best and Worst Performing Course by Completion Rate
print("Best Performing Course:")
course_stats_with_rate.orderBy(desc("completion_rate_%")).limit(1).show()

print("Worst Performing Course:")
course_stats_with_rate.orderBy("completion_rate_%").limit(1).show()

Best Performing Course:
+---------+-----------+--------------+---------------+-------------+-----------------+--------------+
|course_id|course_name|total_enrolled|total_completed|total_dropped|completion_rate_%|dropout_rate_%|
+---------+-----------+--------------+---------------+-------------+-----------------+--------------+
|      109|        NLP|             2|              2|            0|            100.0|           0.0|
+---------+-----------+--------------+---------------+-------------+-----------------+--------------+

Worst Performing Course:
+---------+-----------+--------------+---------------+-------------+-----------------+--------------+
|course_id|course_name|total_enrolled|total_completed|total_dropped|completion_rate_%|dropout_rate_%|
+---------+-----------+--------------+---------------+-------------+-----------------+--------------+
|      108| Blockchain|             2|              0|            2|              0.0|         100.0|
+---------+-----------+---------

In [14]:
# Overall Summary Across All Courses
summary = course_stats_with_rate.agg(
    spark_sum("total_enrolled").alias("overall_enrolled"),
    spark_sum("total_completed").alias("overall_completed"),
    spark_sum("total_dropped").alias("overall_dropped"),
    round(avg("completion_rate_%"), 2).alias("avg_completion_rate_%"),
    round(avg("dropout_rate_%"), 2).alias("avg_dropout_rate_%")
)
print("Overall Summary:")
summary.show()

Overall Summary:
+----------------+-----------------+---------------+---------------------+------------------+
|overall_enrolled|overall_completed|overall_dropped|avg_completion_rate_%|avg_dropout_rate_%|
+----------------+-----------------+---------------+---------------------+------------------+
|              30|               18|             12|                59.17|             40.83|
+----------------+-----------------+---------------+---------------------+------------------+

